In [20]:
#线性回归模型，目标函数y=sum{wi*xi}+b,wi为权重，xi为影响因素，b为偏差因子（单层神经网络）
#平方损失：loss(y,y_estimate)=0.5*(y-y_estimate)^2
#对于训练集X,y求w和b使得0.5*n*sum{y-X*w-b}最小
#梯度下降优化：w(t)=w(t-1)-k*d(loss)/d(w(t-1)),其中k是学习率（步长）
import random
import torch
#w=[2,-3.4],b=4.2
def linreg(X,w,b):
    return torch.matmul(X,w)+b

def data_generate(w,b,n):
    torch.manual_seed(100)
    X=torch.normal(0,1,(n,len(w)))   #n个样本（行）
    y=torch.matmul(X,w)+b
    y+=torch.normal(0,0.01,y.shape)
    return X,y.reshape(-1,1)

def data_iter(n,features,labels):    #从样本空间中生成n个样本
    features_size=len(features)
    indices=list(range(features_size))
    random.shuffle(indices)    #将indices打乱
    for i in range(0,features_size,n):
        batch_indices=torch.tensor(indices[i:min(i+n,features_size)])
        yield features[batch_indices],labels[batch_indices]

def loss(y_hat,y):
    return 0.5*(y_hat-y.reshape(y_hat.shape))**2

def sgd(params,lr,batch_size):
    with torch.no_grad():
        for param in params:
            param-=lr*param.grad/batch_size
            param.grad.zero_()
            
true_w=torch.tensor([2,-3.4])
true_b=4.2
features,labels=data_generate(true_w,true_b,1000)

batch_size=10

w=torch.normal(0,0.01,size=(2,1),requires_grad=True)
b=torch.zeros(1,requires_grad=True)

lr=0.03
epochs=3
for epoch in range(epochs):
    for feature,label in data_iter(batch_size,features,labels):
        l=loss(linreg(feature,w,b),label)
        l.sum().backward()
        sgd([w,b],lr,batch_size)
    with torch.no_grad():
        train_l=loss(linreg(features,w,b),labels)
        print(float(train_l.mean()),w,b,w-true_w,b-true_b,sep='  ')
    


0.03757691755890846  tensor([[ 1.8959],
        [-3.2253]], requires_grad=True)  tensor([4.0148], requires_grad=True)  tensor([[-0.1041,  5.2959],
        [-5.2253,  0.1747]])  tensor([-0.1852])
0.00014019969967193902  tensor([[ 1.9946],
        [-3.3908]], requires_grad=True)  tensor([4.1923], requires_grad=True)  tensor([[-5.3624e-03,  5.3946e+00],
        [-5.3908e+00,  9.2194e-03]])  tensor([-0.0077])
5.5311928008450195e-05  tensor([[ 1.9995],
        [-3.3993]], requires_grad=True)  tensor([4.1997], requires_grad=True)  tensor([[-5.4920e-04,  5.3995e+00],
        [-5.3993e+00,  7.3433e-04]])  tensor([-0.0003])
